In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class MHA_with_KV_cache(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, query: torch.Tensor, key, value, mask=None, kv_cache=None):
        batch_size, seq_len, _ = query.size()

        Q = self.Wq(query).view(batch_size, seq_len, -1, self.head_dim).transpose(1, 2)
        K = self.Wk(key).view(batch_size, seq_len, -1, self.head_dim).transpose(1, 2)
        V = self.Wv(value).view(batch_size, seq_len, -1, self.head_dim).transpose(1, 2)

        if kv_cache is not None:
            if "k" in kv_cache and "v" in kv_cache:
                K = torch.cat((kv_cache["k"], K), dim=2)
                V = torch.cat((kv_cache["v"], V), dim=2)

            kv_cache['k'] = K
            kv_cache['v'] = V

        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(self.head_dim)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attn_weight = F.softmax(scores, dim=-1)

        output = (
            torch.matmul(attn_weight, V)
            .transpose(1, 2)
            .contiguous()
            .view(batch_size, seq_len, self.head_dim * self.num_heads)
        )
        return self.Wo(output)
